# Load Things

In [1]:
import time

In [2]:
# library.py
import pandas as pd
import unicodedata

# diacritics as their unicode value
LOTONE = chr(0x0300)
HITONE = chr(0x0301)
RISETONE = chr(0x030C)
MIDTONE1 = chr(0x0304)
MIDTONE2 = chr(0x0305)
TONECHARS = {LOTONE, HITONE, RISETONE,MIDTONE1,MIDTONE2}

UNDERDOT = chr(0x0323)
UNDERLINE = chr(0x0329)
UNDERDIACS = {UNDERDOT, UNDERLINE}

# split characters into letters (with their diacritics)
def get_letters(text):
    try:
        text = unicodedata.normalize('NFD', text)
    except:
        print(text)
        return []
    text = text.replace(UNDERLINE, UNDERDOT)
    letters = []
    i = 0
    while i < len(text):
        curr_letter = ''
        # look for gb
        if ((i+1) < len(text) and ((text[i] == 'g') and (text[i+1] == 'b'))):
            curr_letter = text[i:i+2]
            i+=2

        # check if next char exists and is a diacritic
        elif ((i+1) < len(text)) and ((text[i+1] in TONECHARS) or text[i+1] in UNDERDIACS):
            if ((i+2) < len(text) and ((text[i+2] in TONECHARS) or text[i+2] in UNDERDIACS)):
                curr_letter = text[i:i+3]
                # print(f"{text[i:i+3]}\t{text[max(i-4, 0):i+4]}\t{text}")
                i+=3 # skip next two chars
            else:
                curr_letter = text[i:i+2]
                i+=2 # skip next char
                
        # normal case (the letter is one single char)
        else: 
            curr_letter = text[i]
            i+=1 # go to next char
        
        # add letter to list
        letters.append(curr_letter.lower())
    return letters

# load in data
def load_dataset(filename):
    return pd.read_csv(filename, header=0, index_col=0)

In [3]:
# syllab.py
from enum import Enum
# from helper import library

# define the possible types of letters
class Letters(Enum):
    SP = 0  # space
    C = 1   # consonant
    M = 2   # m (could be syllabic or consonant)
    N = 3   # n (could be syllabic, consonant, or nasal vowel)
    V = 4   # vowel

# helper functions
def ischar(text):
    return text[0].isalpha()

def chartype(text):
    if ischar(text): 
        if text[0] in ['a', 'e', 'i', 'o', 'u']: return Letters.V
        elif text[0] == 'm': return Letters.M
        elif text[0] == 'n': return Letters.N
        else: return Letters.C
    else: return Letters.SP # punctuation counts as spacing

# syllabify a list of letters, assuming len(letters) != 0
def get_next_syll(letters, syllables, print_it=False):    
    # get types of letters
    # get last char's type
    types = [chartype(letters[-1])]
    
    # get second and third to last chars (if present)
    if len(letters) > 2:
        types.insert(0, chartype(letters[-2]))
        types.insert(0, chartype(letters[-3]))
    # get second to last char, default third to last to SP
    elif len(letters) > 1: 
        types.insert(0, chartype(letters[-2]))
        types.insert(0, Letters.SP)
    # set other values to default value SP
    else:
        types.insert(0, Letters.SP)
        types.insert(0, Letters.SP)
    
    if print_it: print('Types:', types)

    # identify the syllable
    # logic based on Kumolalo 2010
    curr_syll = [] # store current identified syllable
    # handle when last thing isn't a letter
    if (types[-1] == Letters.SP):
        curr_syll = ['SP']
        letters = letters[:-1]
        if print_it: print('SP curr_syll : ', curr_syll)
    # look for CVn (same as DVn in this set up)
    elif (types[-3] in [Letters.C, Letters.M, Letters.N]) and (types[-2] == Letters.V) and (types[-1] == Letters.N):
        curr_syll = letters[-3:]
        letters = letters[:-3]
        if print_it: print('CVn curr_syll : ', curr_syll)
    # look for CV (same as DV)
    elif (types[-2] in [Letters.C, Letters.M, Letters.N]) and (types[-1] == Letters.V):
        curr_syll = letters[-2:]
        letters = letters[:-2]
        if print_it: print('CV curr_syll : ', curr_syll)
    # look for Vn
    elif (types[-2] == Letters.V) and (types[-1] == Letters.N):
        curr_syll = letters[-2:]
        letters = letters[:-2]
        if print_it: print('Vn curr_syll : ', curr_syll)
    # look for V
    elif types[-1] == Letters.V:
        curr_syll = letters[-1:]
        letters = letters[:-1]
        if print_it: print('V curr_syll : ', curr_syll)
    # look for N
    elif types[-1] in [Letters.N, Letters.M]:
        curr_syll = letters[-1:]
        letters = letters[:-1]
        if print_it: print('N curr_syll : ', curr_syll)
    # handle other scenario (must be an error)
    else:
        curr_syll = ['ERR']
        letters = letters[:-1]
        if print_it: print('ERR : ', letters[-3:])

    syllables.insert(0,curr_syll) # add syllable to front of list
    return letters, syllables

def syllabify_letters(letters, print_it = False):
    syllables = []
    while (len(letters) > 0):
        if print_it: print('get_next_syll')
        letters, syllables = get_next_syll(letters, syllables)

    return syllables

def syllabify_df(df):
    syllables = []
    for id, row in df.iterrows():
        letters = get_letters(row['sentence'])
        curr_sylls = syllabify_letters(letters)
        syllables.append([id, curr_sylls])
    return syllables

In [4]:
# Get syllable and word counts
def dup_rows(df):
    print(f'Original Length: {len(df)}')
    dup_sentences = df.duplicated(subset='Sentence', keep=False)
    dup_indices = dup_sentences[dup_sentences == True]
    print(f'{len(dup_indices)} TOTAL DUPLICATES FOUND')
    for i in dup_indices.index:
        print(f'{i}: {df.loc[i]}')
    dropped = df.drop_duplicates(subset='Sentence',keep='first')
    print(f'New Length: {len(dropped)}')
    return dropped


# Prepare Tests

In [5]:
iroyin_full = load_dataset('/Users/graven2/Documents/THESIS/data/iroyinspeech_full2_deduped.csv')
multidiac_full = load_dataset('/Users/graven2/Documents/THESIS/data/multidiac_full.csv')
yad_full = load_dataset('/Users/graven2/Documents/THESIS/data/yad_full_CLEAN.csv')

In [6]:
print('---Iroyin---')
iroyin_syllables = syllabify_df(iroyin_full)
print('---MultiDiac---')
multidiac_syllables = syllabify_df(multidiac_full)
print('---YAD---')
yad_syllables = syllabify_df(yad_full)

---Iroyin---
---MultiDiac---
---YAD---
nan


In [7]:
all_syllables = pd.DataFrame(columns=['Source', 'ID', 'Sentence', 'Syllables'])
print('---IROYIN---')
for row in iroyin_syllables:
    source = 'IroyinSpeech'
    id = row[0]
    sentence = iroyin_full.loc[id, 'sentence']
    syllables = row[1]
    all_syllables.loc[len(all_syllables)] = [source, id, sentence, syllables]

print('---MultiDiac---')
for row in multidiac_syllables:
    source = 'MultiDiac'
    id = row[0]
    sentence = multidiac_full.loc[id, 'sentence']
    syllables = row[1]
    all_syllables.loc[len(all_syllables)] = [source, id, sentence, syllables]

print('---YAD---')
for row in yad_syllables:
    source = 'YAD'
    id = row[0]
    sentence = yad_full.loc[id, 'sentence']
    syllables = row[1]
    all_syllables.loc[len(all_syllables)] = [source, id, sentence, syllables]

---IROYIN---
---MultiDiac---
---YAD---


In [8]:
deduped = dup_rows(all_syllables)

Original Length: 43494
48 TOTAL DUPLICATES FOUND
249: Source                                            IroyinSpeech
ID                                                   00250.wav
Sentence     A ti gbé ìgbésẹ̀ láti bá àwọn tí ọ̀rọ̀ náà kàn...
Syllables    [[a], [SP], [t, i], [SP], [gb, é], [SP], [ì]...
Name: 249, dtype: object
253: Source                                            IroyinSpeech
ID                                                   00254.wav
Sentence     A ó túbọ̀ máa fọwọ́sowọ̀pọ́ pẹ̀lú ìjọba ìpínlẹ...
Syllables    [[a], [SP], [ó], [SP], [t, ú], [b, ọ̀], [SP...
Name: 253, dtype: object
278: Source                                            IroyinSpeech
ID                                                   00279.wav
Sentence     Ọjọ́ kẹ́rìnlélógún, oṣú kejìlá, ọdún yìí ní il...
Syllables    [[ọ], [j, ọ́], [SP], [k, ẹ́], [r, ì, n], [...
Name: 278, dtype: object
312: Source                                            IroyinSpeech
ID                                   

In [9]:
deduped.to_csv('syllabified.csv')

# Run NGrams

In [10]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

In [11]:
# ngrams.py
from enum import Enum

# Create consistent labels for tones
class Tones(Enum):
    H = 1
    M = 0
    L = -1

# All the letters that could have an underdot
DOTCONS = ['s']
DOTVOWELS = ['e', 'o']
DOTLETTERS = ['s', 'e', 'o']

# possible vowels in Yoruba
VOWELS = ['a','e','i','o','u']

# remove the diacritics from a syllable
# can keep 'underdiacs', 'tone', or 'none'
def _rm_diacritics_syll(syllable, keep):
    new_syll = []
    for letter in syllable:
        # corner cases
        if letter == 'SP': return syllable
        if letter == 'ERR': return syllable

        # normal syllable
        include = [letter[0]] # keep original letter
        
        # check second char
        if (len(letter) > 1) and ((letter[1] in UNDERDIACS) and (keep=='underdiacs')):
            include.append(letter[1])
        if (len(letter) > 1) and ((letter[1] in TONECHARS) and (keep=='tone')):
            include.append(letter[1])

        # check third char
        if (len(letter) > 2) and ((letter[2] in UNDERDIACS) and (keep=='underdiacs')):
            include.append(letter[2])
        if (len(letter) > 2) and ((letter[2] in TONECHARS) and (keep=='tone')):
            include.append(letter[2])

        # create string from included characters
        new_syll.append(''.join(include))
    return new_syll

# remove diacritics from a set of syllables
def rm_diacritics_row(row, keep):
    return [_rm_diacritics_syll(x, keep) for x in row['Syllables']]

# remove diacritics from df
# can keep 'underdiacs', 'tone', or 'none'
def rm_diacritics_df(df, keep):
    new_df = df.copy()
    new_df['Syllables'] = new_df.apply(lambda row: rm_diacritics_row(row, keep), axis=1)
    return new_df

# determine which letters get dots
# type = both, vowels, or cons
def _dots_present(syllable, type):
    dots = []
    for index, letter in enumerate(syllable):
        if (letter[0] in DOTCONS) and (type=='both' or type=='cons'):
            # print('DOTCONS', letter, len(letter))
            if (len(letter) > 1) and (letter[1] in UNDERDIACS): dots.append('1')
            elif (len(letter) > 2) and (letter[2] in UNDERDIACS): dots.append('1')
            else: dots.append('0')
        if (letter[0] in DOTVOWELS) and (type=='both' or type=='vowels'):
            # print('DOTVOWELS', letter, len(letter))
            if (len(letter) > 1) and (letter[1] in UNDERDIACS): dots.append('1')
            elif (len(letter) > 2) and (letter[2] in UNDERDIACS): dots.append('1')
            else: dots.append('0')
        if len(dots) <= index: dots.append('0')

    return ' '.join(dots)

# add given dots to a syllable
def _add_dots(syllable, dots):
    new_syll = []
    dots = dots.split(' ')
    # print('STARTING TO ADD DOTS: ', syllable, dots)

    for index, letter in enumerate(syllable):
        if (dots[index] == '1'): 
            curr_char = ''.join([letter, UNDERDOT])
            # print('ADDING TONE')
        else: curr_char = letter
        new_syll.append(curr_char)
        # print(f'index {index}\tdots[index] {dots[index]}\tnew char {curr_char}')
    return new_syll


# find the n-sized context string for syllable i in syllables
def _get_context(syllables, n, i, keep):
    context = []
    for j in range(1, n+1):
        # get -Syl
        # insert at FRONT of list
        if (i-j >= 0): 
            curr_context = syllables[i-j]
            curr_context = _rm_diacritics_syll(curr_context, keep=keep)
            curr_context_str = ''.join(curr_context)
            context.insert(0, curr_context_str)
        else: context.insert(0, '<') # start of sentence token

        # get +Syl
        # add to END of list
        if (i+j < len(syllables)):
            curr_context = syllables[i+j]
            curr_context = _rm_diacritics_syll(curr_context, keep=keep)
            curr_context_str = ''.join(curr_context)
            context.append(curr_context_str)
        else: context.append('>') # end of sentence token
    
    # merge context into a string
    context_str = '.'.join(context) # -Syl.-Syl.+Syl.+Syl
    return context_str


# get n-grams
# n is the number of items before and after to consider
def _syll_grams(syllables, counts, n, keep):
    if not counts: 
        counts = []
        for i in range(n+1): counts.append(dict())
    
    for i, syll in enumerate(syllables):
        syll_str = ''.join(_rm_diacritics_syll(syll, keep=keep))
        # corner case
        if (syll[0] == 'SP') or (syll[0] == 'ERR') or (syll[0] == 'UNK'):
            syll_str = syll[0]

        # counts has format [{syll: {-Syl.-Syl.+Syl.+Syl : {'1 0 0' : count,  '1 1 0' : count, ...}}}, ...]
        # find all contexts in range 0-n (inclusive)
        for j in range(n+1):
            # get contexts for this syllable so far
            poss_contexts = counts[j].get(syll_str, dict())

            # get current context for syllable
            context_str = _get_context(syllables, j, i, keep)

            # update with new dots
            # get previous counts
            curr_dots = _dots_present(syll, 'both')
            context_dots = poss_contexts.get(context_str, dict())
            curr_dot_count = context_dots.get(curr_dots, 0)

            # update all dictionaries
            context_dots.update({curr_dots : curr_dot_count + 1})
            poss_contexts.update({context_str : context_dots})
            counts[j].update({syll_str : poss_contexts})
    return counts

# create a full n-gram count from a df
def create_syll_grams(df, n, keep):
    counts = dict()
    for _, row in df.iterrows():
        counts = _syll_grams(row['Syllables'], counts, n, keep=keep)
    return counts

# predict the dots for a syllable
def pred_dots(syllables, counts, n, keep, print_it=False):
    with_dots = []

    for i, syll in enumerate(syllables):
        # corner case
        if syll[0] == 'SP' or syll[0] == 'ERR':
            with_dots.append(syll)
            continue
        
        syll_str = ''.join(syll)

        # try n, n-1, ..., 0 (and stop as soon as it is possible)
        for j in range(n, -1, -1):
            # get context
            context_str = _get_context(syllables, j, i, keep=keep)

            # collect stored counts
            possible_dots = counts[j].get(syll_str, dict()).get(context_str, dict())

            # find max
            pred_dots = '0 0 0'
            max_use = -1
            for key,value in possible_dots.items():
                if value > max_use:
                    max_use = value
                    pred_dots = key
            syll_with_dots = _add_dots(syll, pred_dots)

            # break if this syllable/sequence of syllables exists in these n-grams
            if not possible_dots: continue # dictionary is empty, so keep trying
            else: 
                if(print_it) : print('n-gram found, stopping; j = ',j,  syll_str, context_str)
                break # dictionary was found, so stop going to smaller syllables

        with_dots.append(syll_with_dots)

    return with_dots

# do full predictions
def predict_all_dots(df, counts, n):
    new_df = df.copy()
    new_df['Prediction'] = new_df.apply(lambda row: pred_dots(row['Syllables'], counts, n), axis=1)
    return new_df

# calculate word error rate for a row, returns (wrong words, total words)
def _eval_row(row, type='both', print_it=False):
    correct = row['Syllables']
    pred = row['Prediction']

    wrong_words = 0
    total_words = 0
    in_word = False # identifies whether currently in a word or not
    curr_word_accurate = True # identifies whether the current word has gotten a tone wrong yet

    # iterate through syllables
    for i in range(len(correct)):
        # check if tones match
        corr_tone = _dots_present(correct[i], type)
        pred_tone = _dots_present(pred[i], type)
        if corr_tone != pred_tone: 
            curr_word_accurate = False

        # check if a word is finished
        if in_word:
            # word has ended
            if correct[i][0] == 'SP':
                in_word = False
                if not curr_word_accurate: 
                    wrong_words += 1
                    if print_it: print('WRONG', correct,pred)
                total_words += 1
                curr_word_accurate = True # reset accuracy
        if correct[i][0] != 'SP': in_word = True
    if in_word:
        total_words+=1
        if not curr_word_accurate: wrong_words+=1

    return pd.Series({'Wrong Words' : wrong_words, 'Total Words' : total_words})

# determine wrong words in df of syllables
def evaluate(df, print_it=False):
    new_df = df.copy()
    new_df[['Wrong Words', 'Total Words']] = new_df.apply(lambda row: _eval_row(row, print_it=print_it), axis=1, result_type='expand')
    return new_df


In [12]:
tester = deduped.loc[4, 'Syllables']
print(tester)

# 29 = ['r', 'ọ̀']
# 2 = ['SP']
# 6 = ['í']
# 22 = ['t', 'u', 'n']
testI = 1
orig_syll = tester[testI]
new_syll = _rm_diacritics_syll(orig_syll, keep='tone')
print(orig_syll, new_syll)

print(_dots_present(orig_syll, 'both'))
print(_dots_present(new_syll, 'both'))

minitester = tester[0:30]
ngram_counts = _syll_grams(minitester, dict(), 4, 'tone')
print(ngram_counts)

untoned = [_rm_diacritics_syll(x, 'tone') for x in minitester]
print(untoned)
toned = pred_dots(untoned, ngram_counts, 4, 'tone')
print(toned)

[['ọ̀'], ['j', 'ọ̀'], ['gb', 'ọ́', 'n'], ['SP'], ['a'], ['j', 'á', 'ń'], ['l', 'é'], ['k', 'o'], ['k', 'ò'], ['SP'], ['s', 'ọ̀'], ['r', 'ọ̀'], ['SP'], ['l', 'á'], ['s', 'ì'], ['k', 'ò'], ['SP'], ['ì'], ['p', 'à'], ['d', 'é'], ['SP'], ['t', 'í'], ['SP'], ['ó'], ['SP'], ['ṣ', 'e'], ['SP']]
['j', 'ọ̀'] ['j', 'ò']
0 1
0 0
[{'ò': {'': {'1': 1}}, 'jò': {'': {'0 1': 1}}, 'gón': {'': {'0 1 0': 1}}, 'SP': {'': {'0': 8}}, 'a': {'': {'0': 1}}, 'jáń': {'': {'0 0 0': 1}}, 'lé': {'': {'0 0': 1}}, 'ko': {'': {'0 0': 1}}, 'kò': {'': {'0 0': 2}}, 'sò': {'': {'0 1': 1}}, 'rò': {'': {'0 1': 1}}, 'lá': {'': {'0 0': 1}}, 'sì': {'': {'0 0': 1}}, 'ì': {'': {'0': 1}}, 'pà': {'': {'0 0': 1}}, 'dé': {'': {'0 0': 1}}, 'tí': {'': {'0 0': 1}}, 'ó': {'': {'0': 1}}, 'se': {'': {'1 0': 1}}}, {'ò': {'<.jò': {'1': 1}}, 'jò': {'ò.gón': {'0 1': 1}}, 'gón': {'jò.SP': {'0 1 0': 1}}, 'SP': {'gón.a': {'0': 1}, 'kò.sò': {'0': 1}, 'rò.lá': {'0': 1}, 'kò.ì': {'0': 1}, 'dé.tí':

In [20]:
# split data
curr_df = deduped
# split into 10 folds, make it even across all three datasets
skf = StratifiedKFold(n_splits=10,shuffle=False)
X = curr_df[['ID', 'Sentence', 'Syllables']].to_numpy()
y = curr_df['Source'].to_numpy() # stratify across the datasets

# run predictions for all folds
WERs = []
for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    # if i > 1: break
    n = 7

    print(f"--- Fold {i} ---")
    train_df = curr_df.iloc[train_index].copy()
    test_df = curr_df.iloc[test_index].copy()

    # create n-grams
    start_time = time.time()
    counts = create_syll_grams(train_df, n=n, keep='tone')
    end_time = time.time()
    print('N-Gram Timing : ', (end_time-start_time), flush=True)

    # create detoned version (to test predictions)
    start_time = time.time()
    detoned = test_df.apply(lambda row: rm_diacritics_row(row, keep='tone'), axis=1)

    # predict tones
    test_df['Prediction'] = detoned.apply(lambda row: pred_dots(row, counts, n=n, keep='tone'))
    end_time = time.time()
    print('Prediction Timing : ', (end_time - start_time), flush=True)

    # evaluate
    start_time = time.time()
    test_df = evaluate(test_df, print_it=False)
    wer = (test_df['Wrong Words'].sum() / test_df['Total Words'].sum()) * 100
    end_time = time.time()
    print('Evaluation Timing : ', (end_time - start_time), flush=True)
    print('WER = ', wer, flush=True)
    WERs.append(wer)

    # print(f"  Train: index={train_index}")
    # print(f"  Test:  index={test_index}")

print(float(sum(WERs)) / float(len(WERs)))

--- Fold 0 ---
N-Gram Timing :  76.68789505958557
Prediction Timing :  3.5402586460113525
Evaluation Timing :  1.2452881336212158
WER =  2.8726389808595845
--- Fold 1 ---
N-Gram Timing :  76.8298192024231
Prediction Timing :  3.800407886505127
Evaluation Timing :  0.49366283416748047
WER =  3.0433068478194945
--- Fold 2 ---
N-Gram Timing :  78.5416932106018
Prediction Timing :  3.5428948402404785
Evaluation Timing :  0.4367060661315918
WER =  3.6706773346362915
--- Fold 3 ---
N-Gram Timing :  78.16047382354736
Prediction Timing :  3.423334836959839
Evaluation Timing :  0.4546208381652832
WER =  4.866294659033263
--- Fold 4 ---
N-Gram Timing :  78.78507018089294
Prediction Timing :  3.54182505607605
Evaluation Timing :  0.44820690155029297
WER =  4.841900853698751
--- Fold 5 ---
N-Gram Timing :  76.97300815582275
Prediction Timing :  4.658783912658691
Evaluation Timing :  0.48050999641418457
WER =  2.814347534588895
--- Fold 6 ---
N-Gram Timing :  78.8038980960846
Prediction Timing :  4